# 03 · Teacher cache  (step 4)
**Accelerator: GPU T4 x1.** Run with **Save Version → Save & Run All (Commit)**; you can close the laptop. Re-running continues where the last run stopped, on any account.


Runs SAUGE, MuGE and MEMO once on the 300 BSDS train+val images (plus a flipped pass that softens the
in-sample maps), packs them, builds the ensembles, and uploads each teacher to HF **as soon as it is done**.
Teachers already on HF are skipped, so if the session ends early just run it again (any account).
The GPU-hours printed at the end count towards the method's training cost.

In [ ]:
# ==== SETTINGS (nothing here needs to change between Kaggle accounts) ====
HF_REPO = "sujalprasad90/pedger-mtkd-store"   # the ONE private Hugging Face repo shared by every account
ACCOUNT = ""                                  # optional label, e.g. your name; sessions are told apart automatically
REPO_URL = ""                                 # optional GitHub URL of the code ("" = Kaggle dataset or HF copy)
# Once per Kaggle account, and for every imported notebook:
#   Add-ons -> Secrets -> HF_TOKEN = a WRITE token from the Hugging Face account that owns HF_REPO, ticked for this notebook
#   Settings -> Internet: On;  Settings -> Accelerator: as written in the title
# The preflight cell below checks all of this and stops with a plain-language fix if something is missing.

In [ ]:
# ---- setup: secrets, code, dependencies (no edits needed) ----
import glob, os, shutil, subprocess, sys, time
SESSION_START = time.time()
from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError("Secret HF_TOKEN is not available: Add-ons -> Secrets -> add HF_TOKEN and tick it for THIS "
                       "notebook, then run again.") from e
os.environ["PEDGER_REMOTE"] = "hf:" + HF_REPO
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub>=0.25", "pyyaml"], check=True)
W, REPO, WS = "/kaggle/working", "/kaggle/working/pedger-mtkd", "/kaggle/working/ws"
if not os.path.exists(REPO):
    attached = sorted(glob.glob("/kaggle/input/*/pedger_kd/__init__.py") + glob.glob("/kaggle/input/*/*/pedger_kd/__init__.py"))
    if attached:                                   # code uploaded as a Kaggle Dataset (pedger-mtkd-code)
        shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), REPO)
        CODE_SOURCE = "kaggle dataset"
    elif REPO_URL:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
        CODE_SOURCE = "git"
    else:                                          # copy pushed to Hugging Face by notebook 01
        from huggingface_hub import snapshot_download
        snapshot_download(HF_REPO, repo_type="dataset", allow_patterns=["code/*"], local_dir=f"{W}/hf_code",
                          token=os.environ["HF_TOKEN"])
        shutil.copytree(f"{W}/hf_code/code", REPO)
        CODE_SOURCE = "huggingface"
    print("code from:", CODE_SOURCE)
os.chdir(REPO)
sys.path.insert(0, REPO)
!python tools/fetch_upstream.py --teachers
# preflight: session type, GPUs, internet, Hugging Face account and repo access
if subprocess.run([sys.executable, "-m", "pedger_kd.preflight"]).returncode:
    raise RuntimeError("Preflight failed: fix the PROBLEM lines printed above, then run the notebook again.")

In [ ]:
from pedger_kd.sync import open_remote
remote = open_remote(os.environ["PEDGER_REMOTE"])
remote.download("data/bsds", WS)
remote.download("teachers/bsds", WS)
SPLIT, TW, TC = "trainval", f"{W}/work", f"{WS}/teachers/bsds"
def on_hf(t): return remote.exists(f"teachers/bsds/{SPLIT}/{t}.npy")
def push(t):
    remote.upload({f"teachers/bsds/{SPLIT}/{t}.npy": f"{TC}/{SPLIT}/{t}.npy",
                   f"teachers/bsds/{SPLIT}/{t}.json": f"{TC}/{SPLIT}/{t}.json"}, f"teacher {t}")
todo = [t for t in ["sauge", "muge", "memo"] if not on_hf(t)]
print("teachers still to run:", todo)
if todo:
    !python teachers/export_images.py --store {WS}/data/bsds --split {SPLIT} --out {TW}/img --flip

In [ ]:
if "sauge" in todo:
    !pip install -q einops
    !wget -q -nc https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -P {W}
    for sfx in ["", "_flip"]:
        !python teachers/run_sauge.py --images {TW}/img{sfx} --out {TW}/sauge{sfx} --sam-ckpt {W}/sam_vit_b_01ec64.pth
    !python teachers/pack_cache.py --store {WS}/data/bsds --split {SPLIT} --teacher sauge --pred-dir {TW}/sauge --pred-dir-flip {TW}/sauge_flip --out {TC}
    push("sauge")

In [ ]:
if "muge" in todo:
    !pip install -q efficientnet_pytorch gdown
    !gdown -q 15NucsEeHAFwo5O2s11pMR1BikoUtiuUX -O {W}/muge_bsds.pth
    for sfx in ["", "_flip"]:
        !python teachers/run_muge.py --images {TW}/img{sfx} --out {TW}/muge{sfx} --ckpt {W}/muge_bsds.pth
    !python teachers/pack_cache.py --store {WS}/data/bsds --split {SPLIT} --teacher muge --pred-dir {TW}/muge --pred-dir-flip {TW}/muge_flip --out {TC}
    push("muge")

In [ ]:
if "memo" in todo:
    %cd third_party/MEMO_Edge_Detection
    !pip install -q -r requirements.txt
    !python download_checkpoints.py --model bsds-late --model synthetic-late
    for sfx in ["", "_flip"]:
        !python edge_prediction.py --test_folder {TW}/img{sfx} --save_folder {TW}/memo{sfx} --config_file configs/discrete_BSDS_finetune/binary_lora_default.yaml --model_path pretrained_models/MEMO_BSDS_ft_late/mp_rank_00_model_states.pt --base_model_path pretrained_models/MEMO_synthetic_late/mp_rank_00_model_states.pt --guidance_scale 1.4 --max_steps 20
    %cd /kaggle/working/pedger-mtkd
    !python teachers/pack_cache.py --store {WS}/data/bsds --split {SPLIT} --teacher memo --pred-dir {TW}/memo/prediction --pred-dir-flip {TW}/memo_flip/prediction --out {TC}
    push("memo")

In [ ]:
# ensembles (trainval) + the same maps cut out for the dev split (train) used by the ablations
remote.download("teachers/bsds", WS)
ALL = "sauge muge memo ensemble_mean ensemble_confidence"
for mode in ["mean", "confidence"]:
    !python teachers/ensemble.py --teacher-root {TC} --split {SPLIT} --teachers sauge muge memo --mode {mode}
!python teachers/subset_cache.py --store {WS}/data/bsds --teacher-root {TC} --from trainval --to train --teachers {ALL}
!python -m pedger_kd.sync push-dir --local {TC} --to teachers/bsds
print(f"teacher caching this session: {(time.time() - SESSION_START) / 3600:.2f} GPU-h")

In [ ]:
# step 6 gate: red = teacher, green = label after the joint augmentation; they must trace the same contours
!python data/visualize_samples.py --store {WS}/data/bsds --split {SPLIT} --n 8 --teacher-root {TC} --teacher ensemble_mean --augmented --out {W}/checks/aug_align.png
from IPython.display import Image, display; display(Image(f"{W}/checks/aug_align.png"))

In [ ]:
# keep the notebook output small (everything important is already on Hugging Face)
for d in ["ws/data", "ws/teachers", "ws/.staging", "hf_code", "work", "BSR"]:
    shutil.rmtree(f"/kaggle/working/{d}", ignore_errors=True)
print(f"session used {(time.time() - SESSION_START) / 3600:.2f} h")